In [1]:
import numpy as np 
import pandas as pd

In [2]:
movies = pd.read_csv('tmdb_5000_credits.csv')
credits = pd.read_csv('tmdb_5000_movies.csv')

In [3]:
movies = movies.merge(credits,on = 'title')

In [4]:
movies.head(1)

,movie_id,title,cast,crew,budget,genres,homepage,id,keywords,original_language,...,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,...,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,7.2,11800


In [5]:
movies = movies[['movie_id','overview','cast','crew','genres','title','keywords']]

In [6]:
movies.isnull().sum()

movie_id    0
overview    3
cast        0
crew        0
genres      0
title       0
keywords    0
dtype: int64

In [7]:
movies = movies.dropna()

In [8]:
movies.duplicated().sum()

np.int64(0)

In [9]:
movies['overview'].isna().sum()

np.int64(0)

In [10]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [11]:
import ast
def convert(obj):
    List = []
    for i in ast.literal_eval(obj):  # it is a string we need to convert it into list thats why ast.literal_eval
        List.append(i['name'])       # to add only the genre name to the list 
    return List

In [12]:
movies['genres'] = movies['genres'].apply(convert)

In [13]:
movies['keywords'] = movies['keywords'].apply(convert)

In [14]:
def fetch_cast_top3(obj):
    List = []
    counter = 0
    for i in ast.literal_eval(obj):  
        if(counter != 3):              # considering only the top three actor/actress
            List.append(i['name'])
            counter += 1
        else:
            break
    return List

In [15]:
movies['cast'] = movies['cast'].apply(fetch_cast_top3)

In [16]:
movies.head()

,movie_id,overview,cast,crew,genres,title,keywords
0,19995,"In the 22nd century, a paraplegic Marine is di...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...","[Action, Adventure, Fantasy, Science Fiction]",Avatar,"[culture clash, future, space war, space colon..."
1,285,"Captain Barbossa, long believed to be dead, ha...","[Johnny Depp, Orlando Bloom, Keira Knightley]","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...","[Adventure, Fantasy, Action]",Pirates of the Caribbean: At World's End,"[ocean, drug abuse, exotic island, east india ..."
2,206647,A cryptic message from Bond’s past sends him o...,"[Daniel Craig, Christoph Waltz, Léa Seydoux]","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...","[Action, Adventure, Crime]",Spectre,"[spy, based on novel, secret agent, sequel, mi..."
3,49026,Following the death of District Attorney Harve...,"[Christian Bale, Michael Caine, Gary Oldman]","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...","[Action, Crime, Drama, Thriller]",The Dark Knight Rises,"[dc comics, crime fighter, terrorist, secret i..."
4,49529,"John Carter is a war-weary, former military ca...","[Taylor Kitsch, Lynn Collins, Samantha Morton]","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...","[Action, Adventure, Science Fiction]",John Carter,"[based on novel, mars, medallion, space travel..."


In [17]:
def fetch_director(obj):
    List = []
    for i in ast.literal_eval(obj):  
        if(i['job'] == 'Director'):
            List.append(i['name'])
    return List

In [18]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [19]:
movies.head(1)

,movie_id,overview,cast,crew,genres,title,keywords
0,19995,"In the 22nd century, a paraplegic Marine is di...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],"[Action, Adventure, Fantasy, Science Fiction]",Avatar,"[culture clash, future, space war, space colon..."


In [20]:
# to remove spaces so that it consider ScienceFiction as a single tag not science and fiction as seperate
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x]) 
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])

In [21]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [40]:
movies['tag'] = movies['overview'] + movies['crew'] + movies['cast'] + movies['keywords'] + movies['genres'] 

In [41]:
movies.head(1)

,movie_id,overview,cast,crew,genres,title,keywords,tag
0,19995,"[In, the, 22nd, century,, a, paraplegic, Marin...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[Action, Adventure, Fantasy, ScienceFiction]",Avatar,"[cultureclash, future, spacewar, spacecolony, ...","[In, the, 22nd, century,, a, paraplegic, Marin..."


In [70]:
movies_df = movies[['movie_id','title','tag']]
movies_df['tag'] = movies_df['tag'].apply(lambda x:" ".join(x))
movies_df.head()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16284\3383464475.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_df['tag'] = movies_df['tag'].apply(lambda x:" ".join(x))


,movie_id,title,tag
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [75]:
movies_df['tag'] = movies_df['tag'].str.lower()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16284\2121835774.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_df['tag'] = movies_df['tag'].str.lower()


In [76]:
movies_df.head()

,movie_id,title,tag
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [77]:
import nltk

In [78]:
# Stemming
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [79]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
        
    return " ".join(y)

In [80]:
movies_df['tag'] = movies_df['tag'].apply(stem)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16284\1950200779.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_df['tag'] = movies_df['tag'].apply(stem)


In [81]:
movies_df['tag']

0       in the 22nd century, a parapleg marin is dispa...
1       captain barbossa, long believ to be dead, ha c...
2       a cryptic messag from bond’ past send him on a...
3       follow the death of district attorney harvey d...
4       john carter is a war-weary, former militari ca...
                              ...                        
4804    el mariachi just want to play hi guitar and ca...
4805    a newlyw couple' honeymoon is upend by the arr...
4806    "signed, sealed, delivered" introduc a dedic q...
4807    when ambiti new york attorney sam is sent to s...
4808    ever sinc the second grade when he first saw h...
Name: tag, Length: 4806, dtype: object

In [86]:
#converting most frequent words to vector 
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 5000, stop_words = 'english')

In [91]:
vectors = cv.fit_transform(movies_df['tag']).toarray()

In [96]:
vectors.shape

(4806, 5000)

In [95]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [102]:
# distance between each vector 
from sklearn.metrics.pairwise import cosine_similarity

In [103]:
similarity = cosine_similarity(vectors)

In [125]:
def recommend(movie_name):
    movie_index = movies_df[movies_df['title'] == movie_name].index[0]   
    movie_list = sorted(list(enumerate(similarity[movie_index])),reverse = True, key = lambda x:x[1])[1:6] 

    for i in movie_list:
        print(movies_df.iloc[i[0]].title)

# index gives all the indices that matches cond, index[0] will give the very first movie that matches condn
# enumerate to maintain the index even after sorting 
# x[1] to sort on the basis of similarity score
# [1:6] first five except the movie we looking for 
# i[0] will give the index 
    

In [126]:
recommend('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [128]:
import pickle 
pickle.dump(movies_df.to_dict(),open('movie_dict.pkl','wb'))    
# write binary
# need to convert it to dictionary can't transfer pd array 

In [130]:
 pickle.dump(vectors,open('vectors.pkl','wb'))  